In [0]:
file_location = "/FileStore/tables/Files/2011_summary.csv"
df = spark.read.format("csv") \
  .option("inferSchema", "false") \
  .option("header", 'true') \
  .option("sep", ",") \
  .load(file_location)

display(df)

DEST_COUNTRY_NAME,ORIGIN_COUNTRY_NAME,count
United States,Saint Martin,2
United States,Guinea,2
United States,Croatia,1
United States,Romania,3
United States,Ireland,268
Egypt,United States,13
United States,India,76
United States,Singapore,24
United States,Grenada,59
Costa Rica,United States,494


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, when, explode, regexp_replace, regexp_extract, array_contains, split
import pandas as pd
import numpy as np
df_filled = df.fillna({"DEST_COUNTRY_NAME": "Unknown", "ORIGIN_COUNTRY_NAME": "Unknown", "count": "0"})
#df_exploded = df_filled.withColumn("Exploded", explode(array_contains(col("DEST_COUNTRY_NAME"), "United States")))
df_dropped = df.dropna()
df_cleaned = df.withColumn("DEST_COUNTRY_CLEAN", regexp_replace(col("DEST_COUNTRY_NAME"), "United", "U."))
#display(df_cleaned)
df_ex = df.withColumn("DEST_COUNTRY_EX", regexp_extract(col("DEST_COUNTRY_NAME"), "(^.)", 1))
df_array = df.withColumn("Country_Array", split(col("DEST_COUNTRY_NAME"), " "))
df_con=df_array.withColumn("IF is", array_contains(col("Country_Array"), "United"))
df_exploded = df_array.withColumn("Exploded_Country", explode(col("Country_Array")))
display(df_exploded)

# Pandas: ifnull i nullIf
pdf = pd.DataFrame({"col1": [1, np.nan, 3], "col2": [np.nan, 2, 3]})
pdf["col1_filled"] = pdf["col1"].fillna(0)  # Odpowiednik ifnull
pdf["col3"] = np.where(pdf["col1"] == pdf["col2"], np.nan, pdf["col1"])  # Odpowiednik nullIf

# Zamiana wartości w Pandas
pdf["col1_replaced"] = pdf["col1"].replace(1, 100)

DEST_COUNTRY_NAME,ORIGIN_COUNTRY_NAME,count,Country_Array,Exploded_Country
United States,Saint Martin,2,"List(United, States)",United
United States,Saint Martin,2,"List(United, States)",States
United States,Guinea,2,"List(United, States)",United
United States,Guinea,2,"List(United, States)",States
United States,Croatia,1,"List(United, States)",United
United States,Croatia,1,"List(United, States)",States
United States,Romania,3,"List(United, States)",United
United States,Romania,3,"List(United, States)",States
United States,Ireland,268,"List(United, States)",United
United States,Ireland,268,"List(United, States)",States


In [0]:
# Agregacje
agg_df = df.groupBy("DEST_COUNTRY_NAME").agg(
    F.count("count").alias("flight_count"),  # Liczba wystąpień danego kraju jako destynacji
    F.sum(col("count").cast("int")).alias("total_passengers"),  # Suma pasażerów do danego kraju
    F.avg(col("count").cast("int")).alias("avg_passengers")  # Średnia liczba pasażerów
)
display(agg_df)


DEST_COUNTRY_NAME,flight_count,total_passengers,avg_passengers
Anguilla,1,21,21.0
Russia,1,199,199.0
Paraguay,1,85,85.0
Yemen,1,1,1.0
Senegal,1,29,29.0
Sweden,1,59,59.0
Kiribati,1,28,28.0
Guyana,1,26,26.0
Philippines,1,127,127.0
Malaysia,1,2,2.0


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, DoubleType

def convert_to_double(value):
    try:
        return float(value)
    except ValueError:
        return None
convert_to_double_udf = udf(convert_to_double, DoubleType())
df = df.withColumn("count_double", convert_to_double_udf(col("count")))
display(df)


DEST_COUNTRY_NAME,ORIGIN_COUNTRY_NAME,count,count_double
United States,Saint Martin,2,2.0
United States,Guinea,2,2.0
United States,Croatia,1,1.0
United States,Romania,3,3.0
United States,Ireland,268,268.0
Egypt,United States,13,13.0
United States,India,76,76.0
United States,Singapore,24,24.0
United States,Grenada,59,59.0
Costa Rica,United States,494,494.0


In [0]:
def uppercase_udf(country_series: pd.Series) -> pd.Series:
    return country_series.str.upper()
df = df.withColumn("count_double", col("count").cast("double"))
display(df)


DEST_COUNTRY_NAME,ORIGIN_COUNTRY_NAME,count,count_double
United States,Saint Martin,2,2.0
United States,Guinea,2,2.0
United States,Croatia,1,1.0
United States,Romania,3,3.0
United States,Ireland,268,268.0
Egypt,United States,13,13.0
United States,India,76,76.0
United States,Singapore,24,24.0
United States,Grenada,59,59.0
Costa Rica,United States,494,494.0
